In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
from rapidfuzz import process
import pickle

In [2]:
# Count number of species per aviary 
metadata_path = 'metadata_aviaries/'
species_df = pd.read_csv(metadata_path + 'Avieries_obsolete.csv', encoding='latin-1')

number_species = species_df.groupby('Aviary')['Common name'].nunique()
number_individuals = species_df.groupby('Aviary')['Total count'].sum()
metadata_paths = species_df.groupby('Aviary')['preprocessed data'].first()  

combined_df = pd.DataFrame({'Number of Species': number_species, 'Number of Individuals': number_individuals, "preprocessed data": metadata_paths})
combined_df = combined_df.dropna() # Some aviaries don't have metadata files, so we drop those rows for now

# Link preprocessed data path to metadata path as they are not in the same format
aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
for idx, row in combined_df.iterrows():
    avirary_path = row["preprocessed data"]
    best_match = process.extractOne(avirary_path, aviaries_metadata)
    if best_match and best_match[1] > 80:
        file_path = best_match[0]

    combined_df.loc[row.name, "Metadata_filename"] = file_path

combined_df.reset_index(inplace=True)
combined_df


/tmp/ipykernel_3370/2565535384.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'fl_3s_avifauna_flamingos_sept25_data_meta.xlsx' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  combined_df.loc[row.name, "Metadata_filename"] = file_path


,Aviary,Number of Species,Number of Individuals,preprocessed data,Metadata_filename
0,"Avifauna, Cuba aviary",8,247,fl_avifauna_flamingos_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
1,"Avifauna, Umvikeli aviary",13,71,fl_avifauna_vultures_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
2,"Avifauna, storage",4,36,fl_avifauna_storage_sept2025_data,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
3,"Beekse Bergen, Aviary 2",1,2,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
4,"Beekse Bergen, Aviary 3",8,163,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
5,"Beekse Bergen, Aviary 4 (Africa)",8,182,fl_beekse_bergen_20250412,fl_beekse_bergen_20250412_meta.xlsx
6,"Beekse Bergen, aviary 5",2,113,fl_beekse_bergen_flaminogos_sept2025_data,fl_beekse_bergen_20250412_meta.xlsx
7,"Blijdorp, Flamingos",4,204,fl_blijdorp_flamingos_dec2025_data,fl_blijdorp_flamingos_dec2025_data_meta.xlsx
8,"Cologne, Flamingos",1,153,fl_cologne_zoo_flamingos_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx
9,"Cologne, Hippodom",17,98,fl_cologne_zoo_indoors_long_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx


In [9]:
# count number of bird call per aviary meta data file
# We don't have all the metadata files, so we can't get the info for all aviaries in the cell above

aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
EDA_df = pd.DataFrame(columns=["file_name", "call_count", "total_entries", "Ratio", "Number_Species", "Number_Individuals", "start_date", "end_date", "Duration_days", "Unique_days"])

for file in aviaries_metadata:
    print(f"Processing file: {file}")

    try:
        df = pd.read_excel(metadata_path + file)

        call_count = df["fusion_model_prediction"].notnull().sum()
        
        number_species = combined_df[combined_df["Metadata_filename"] == file]["Number of Species"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
        number_individuals = combined_df[combined_df["Metadata_filename"] == file]["Number of Individuals"].iloc[0] if file in combined_df["Metadata_filename"].values else 0

        start_date = df["datetime"].min()
        end_date = df["datetime"].max()
        unique_days = df["datetime"].dt.date.nunique()

        new_row = {
            "file_name": file,
            "call_count": call_count,
            "total_entries": len(df),
            "Ratio": call_count / len(df),
            "Number_Species": number_species,
            "Number_Individuals": number_individuals,
            "start_date": start_date,
            "end_date": end_date,
            "Duration_days": (end_date - start_date).days,
            "Unique_days": unique_days
        }
        
        EDA_df.loc[len(EDA_df)] = new_row

    except Exception as e:
        print(f"Error processing file {file}: {e}")
            
EDA_df

Processing file: fl_zoo_eindhoven_20250308_meta.xlsx
Processing file: fl_zoo_eindhoven_20250426_meta.xlsx
Processing file: fl_gaia_zoo_savannah_08aug25_data_meta.xlsx
Processing file: fl_gaia_zoo_congo_15aug25_data_meta.xlsx
Processing file: fl_gaia_zoo_taiga_18Jul25_data_meta.xlsx
Processing file: fl_avifauna_vultures_prox_sept25_data_meta.xlsx
Processing file: fl_zoo_eindhoven_20250503_meta.xlsx
Processing file: fl_beekse_bergen_20250412_meta.xlsx
Processing file: fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx
Processing file: fl_3s_avifauna_flamingos_sept25_data_meta.xlsx
Processing file: fl_blijdorp_flamingos_dec2025_data_meta.xlsx
Processing file: fl_zoo_eindhoven_20250315_meta.xlsx
Processing file: fl_beekse_bergen_20250404_meta.xlsx
Processing file: fl_zoo_helsinki_20250624_meta.xlsx
Processing file: fl_zoo_parc_aug25_data_meta.xlsx
Processing file: fl_avifauna_storage_ibises_sept25_data_meta.xlsx
Processing file: fl_zoo_helsinki_20250701_meta.xlsx


,file_name,call_count,total_entries,Ratio,Number_Species,Number_Individuals,start_date,end_date,Duration_days,Unique_days
0,fl_zoo_eindhoven_20250308_meta.xlsx,65440,74420,0.879334,10,216,2025-03-08 12:09:35,2025-03-15 12:12:17,7,8
1,fl_zoo_eindhoven_20250426_meta.xlsx,82988,97864,0.847993,0,0,2025-04-26 11:35:26,2025-05-03 11:34:33,6,8
2,fl_gaia_zoo_savannah_08aug25_data_meta.xlsx,108628,126165,0.860999,12,209,2025-07-24 14:00:05,2025-08-08 10:08:28,14,11
3,fl_gaia_zoo_congo_15aug25_data_meta.xlsx,27147,42872,0.633210,3,68,2025-08-08 11:00:02,2025-08-15 07:55:02,6,8
4,fl_gaia_zoo_taiga_18Jul25_data_meta.xlsx,26295,34307,0.766462,23,102,2025-07-18 12:00:05,2025-07-24 13:16:39,6,7
5,fl_avifauna_vultures_prox_sept25_data_meta.xlsx,19985,28980,0.689614,0,0,2025-09-12 13:36:37,2025-09-20 11:11:42,7,9
6,fl_zoo_eindhoven_20250503_meta.xlsx,16465,29901,0.550650,3,21,2025-05-03 11:39:08,2025-05-10 11:52:07,7,8
7,fl_beekse_bergen_20250412_meta.xlsx,33816,50560,0.668829,8,182,2025-04-12 13:16:29,2025-04-19 12:44:38,6,8
8,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx,131557,148163,0.887921,1,153,2025-11-10 11:53:32,2025-11-26 06:23:17,15,17
9,fl_3s_avifauna_flamingos_sept25_data_meta.xlsx,117833,167919,0.701725,8,247,2025-09-06 15:11:18,2025-09-12 12:57:44,5,7


In [10]:
EDA_df.to_csv("Aviary_EDA.csv", index=False)